# TRICE — Business Entity Resolution (Kaggle runner)

**Team Vortex** · Aryan Bansal (lead) · Sachin Kumar · Daksh Tandon · Parth Aggarwal

Runs the full pipeline on Kaggle: **mine → prepare → train → tune → infer**, then writes
`matching_results.tsv` and `candidate_pairs.tsv` to `/kaggle/working/output/`.

### Before you run
1. **Add the dataset** (right panel → *Add Input* → *Datasets*): upload the seven challenge
   TSVs. Any folder layout works — the notebook searches `/kaggle/input` for them by name
   and ignores macOS `._` sidecar files.
2. **Settings**: Accelerator **None** (models are gradient-boosted trees), Internet **On**
   (for the git clone + pip).
3. **Run All.** Keep `SUBSET = 4000` (config cell) for a fast end-to-end check first, then
   set `SUBSET = None` and re-run for the real full-test submission (~1.5–2.5 h).

Every stage streams its output; on failure the real traceback is printed and also saved to
`artifacts/last_error.txt`.

In [ ]:
# --- 1. clone / update the code -------------------------------------------------
import os, subprocess, sys

REPO = "https://github.com/iittjjee2024/TRICE.git"
CODE = "/kaggle/working/TRICE"
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "--depth", "1", REPO, CODE], check=True)
else:
    # hard-reset to origin/main so the notebook always runs the latest committed code,
    # even if a previous run left local changes that would block a fast-forward pull.
    subprocess.run(["git", "-C", CODE, "fetch", "origin", "main"], check=False)
    subprocess.run(["git", "-C", CODE, "reset", "--hard", "origin/main"], check=False)
head = subprocess.run(["git", "-C", CODE, "rev-parse", "--short", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
print("code at", CODE, "| commit", head)

In [ ]:
# --- 2. dependencies ------------------------------------------------------------
# Kaggle already ships numpy/pandas/scipy/scikit-learn/pyarrow/lightgbm; we only add the
# two light packages the pipeline needs, leaving the rest to Kaggle's stack.
!pip install -q rapidfuzz Unidecode
import numpy, pandas, sklearn, scipy, rapidfuzz, pyarrow, unidecode
try:
    import lightgbm; lgbm = lightgbm.__version__
except Exception as e:
    lgbm = f"unavailable ({e}) — will fall back to HistGradientBoosting"
print("numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| sklearn", sklearn.__version__, "| lightgbm", lgbm)

In [ ]:
# --- 3. configuration + locate the dataset --------------------------------------
import glob

WORK = "/kaggle/working/trice_run"        # writable: dataset links, store, model
OUT  = "/kaggle/working/output"           # the two submission TSVs land here
os.makedirs(WORK, exist_ok=True); os.makedirs(OUT, exist_ok=True)

# Fast smoke run vs full run. Set SUBSET = None for the real submission.
SUBSET = 4000                             # entities/country for training
TRAIN_ENTITIES = 70000 if SUBSET is None else SUBSET

# If auto-detection ever fails, set this to the folder that holds the 7 TSVs.
DATA_DIR_OVERRIDE = None

REQUIRED = [("train", "train_source1.tsv"), ("train", "train_source2.tsv"),
            ("train", "train_source3.tsv"), ("train", "train_ground_truth.tsv"),
            ("test", "test_source1.tsv"), ("test", "test_source2.tsv"),
            ("test", "test_source3.tsv")]

def _is_junk(path):
    # macOS zips add a tiny '._name' AppleDouble sidecar and a __MACOSX/ folder; matching
    # one of those (254 bytes of metadata) instead of the real file breaks everything.
    base = os.path.basename(path)
    return base.startswith("._") or "__MACOSX" in path.replace("\\", "/").split("/")

def _norm(name):
    return os.path.basename(name).lower().lstrip("-_ ")

all_tsv = [os.path.join(r, f) for r, _d, fs in os.walk("/kaggle/input")
           for f in fs if f.lower().endswith(".tsv") and not _is_junk(os.path.join(r, f))]
found = {}
for split, fname in REQUIRED:
    exact = [p for p in all_tsv if os.path.basename(p).lower() == fname.lower()]
    hits = exact or [p for p in all_tsv if _norm(p) == fname.lower()]
    hits = [p for p in hits if os.path.getsize(p) > 1024] or hits
    if hits:
        found[(split, fname)] = sorted(hits, key=os.path.getsize, reverse=True)[0]
missing = [f"{s}/{f}" for (s, f) in REQUIRED if (s, f) not in found]
if missing:
    print("MISSING:", missing)
    print("real .tsv files under /kaggle/input (macOS sidecars excluded):")
    for p in all_tsv:
        print(f"  {os.path.getsize(p)/1e6:8.1f} MB  {p}")
    raise SystemExit("Attach the dataset via 'Add Input' — it must contain the 7 TSVs.")

# Assemble a clean train/ + test/ layout in the writable dir via symlinks, so the pipeline
# sees exactly what it expects regardless of how the dataset was uploaded.
DATA_DIR = DATA_DIR_OVERRIDE or os.path.join(WORK, "dataset")
if not DATA_DIR_OVERRIDE:
    for (split, fname), src in found.items():
        d = os.path.join(DATA_DIR, split); os.makedirs(d, exist_ok=True)
        dst = os.path.join(d, fname)
        if os.path.islink(dst) or os.path.exists(dst):
            os.remove(dst)
        os.symlink(os.path.realpath(src), dst)
print("selected dataset files:")
for (s, f), p in sorted(found.items()):
    print(f"  {s}/{f}: {os.path.getsize(p)/1e6:8.1f} MB")
print("DATA_DIR =", DATA_DIR)

In [ ]:
# --- 4. wire pipeline paths via environment variables (no symlinks into the repo) --
# The scripts honour these three vars (src/trice/paths.py); we point them at the writable
# Kaggle locations so nothing needs to be written inside the read-only input mount.
ART = os.path.join(WORK, "artifacts"); os.makedirs(ART, exist_ok=True)
os.environ["TRICE_DATA_DIR"] = DATA_DIR
os.environ["TRICE_ARTIFACTS_DIR"] = ART
os.environ["TRICE_OUTPUT_DIR"] = OUT

sys.path.insert(0, os.path.join(CODE, "src"))
import importlib, trice.paths
importlib.reload(trice.paths)
print("TRICE_DATA_DIR      =", trice.paths.dataset_dir(CODE))
print("TRICE_ARTIFACTS_DIR =", trice.paths.artifacts_dir(CODE))
print("TRICE_OUTPUT_DIR    =", trice.paths.output_dir(CODE))

In [ ]:
# --- 5. clean any stale record store --------------------------------------------
# A previous broken run (e.g. one that read a macOS sidecar) could have written empty
# Parquet files, which would silently starve training ('train countries = []'). Removing
# the store forces stage 03 to rebuild it from the real files.
import shutil
store = os.path.join(ART, "store")
if os.path.isdir(store):
    shutil.rmtree(store, ignore_errors=True)
    print("removed stale store", store)
else:
    print("no existing store — clean start")

In [ ]:
# --- 6. preflight: exercise the pipeline's OWN tolerant readers IN-KERNEL --------
# Reads go through trice.paths.open_text (utf-8 with errors='replace'), so the stray
# non-UTF-8 bytes in the dataset do not crash anything. If this passes, the stages run.
import trice.normalize, trice.records, trice.evaluate
for _m in (trice.paths, trice.normalize, trice.records, trice.evaluate):
    importlib.reload(_m)
from trice.paths import open_text
from trice.records import iter_blocks
import inspect
assert "errors" in inspect.getsource(open_text), \
    "cloned code is stale — re-run the clone cell (git reset should give the latest)"

gt = os.path.join(DATA_DIR, "train", "train_ground_truth.tsv")
with open_text(gt) as fh:
    print("ground truth header:", next(fh).rstrip().split("\t"))
s1 = os.path.join(DATA_DIR, "train", "train_source1.tsv")
parts = next(iter_blocks(s1, block_lines=5))[0].rstrip("\n").split("\t")
print("source1 columns:", len(parts), parts[:2])
nm = trice.normalize.normalize_name(parts[1])
ad = trice.normalize.normalize_address(parts[2])
print("normalise OK:", nm.core(), "|", ad.alpha_tokens[:4])
open(os.path.join(ART, "_wtest"), "w").close(); os.remove(os.path.join(ART, "_wtest"))
print("\nPREFLIGHT OK — stages below should run.")

In [ ]:
# --- 7. stage runner ------------------------------------------------------------
# Streams each stage live; on failure reprints the tail and the saved traceback so the
# real error is always visible (a subprocess would otherwise hide it).
import time, collections

def run(args, title):
    print("=" * 78, f"\n{title}\n" + "=" * 78, flush=True)
    t0 = time.time(); tail = collections.deque(maxlen=40)
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8"}
    p = subprocess.Popen([sys.executable, *args], cwd=CODE, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end=""); tail.append(line)
    p.wait()
    print(f"\n[{title}] exit={p.returncode}  {time.time()-t0:.0f}s", flush=True)
    if p.returncode != 0:
        print("\n----- last lines of failing stage -----\n" + "".join(tail))
        err = os.path.join(ART, "last_error.txt")
        if os.path.isfile(err):
            print("\n----- full traceback (artifacts/last_error.txt) -----")
            print(open(err, encoding="utf-8").read())
        raise RuntimeError(f"{title} failed (exit {p.returncode})")

In [ ]:
# --- 8. correctness tests (fast, need no dataset) -------------------------------
run(["scripts/test_decide.py"], "test: decision-layer proofs")
run(["scripts/test_union.py"], "test: candidate union")

In [ ]:
# --- 9. stage 1: mine token aliases from the training ground truth --------------
run(["scripts/02_mine_variants.py", "--sample", "250000"], "mine variants")

In [ ]:
# --- 10. stage 2: normalise all records into the Parquet record store -----------
# Kaggle CPU has ~4 cores; use 3 workers. --force rebuilds even if an earlier run left an
# empty store. This stage will print real row counts (e.g. train/source1: 2,206,821 rows).
run(["scripts/03_prepare.py", "--workers", "3", "--force"], "prepare record store")

In [ ]:
# --- 11. stage 3: train the matcher + score a held-out validation split ---------
run(["scripts/05_train.py", "--entities", str(TRAIN_ENTITIES), "--run-id", "kaggle"],
    "train + validate")

In [ ]:
# --- 12. stage 4: search the decision-layer configuration on validation ---------
run(["scripts/07_tune_decision.py", "--run-id", "kaggle"], "tune decision")

In [ ]:
# --- 13. inspect the validation metrics -----------------------------------------
import json
m = json.load(open(os.path.join(ART, "runs", "kaggle", "metrics.json")))
print("model:", m["model_kind"], "| use_stage2:", m["use_stage2"])
print("stage1:", m["stage1"]); print("stage2:", m["stage2"])
ef = m["decision_rules"]["expected_f"]
print(f"VAL macro F0.5 = {ef['val_macro_f05']:.5f}  "
      f"P={ef['val_macro_precision']:.4f}  R={ef['val_macro_recall']:.4f}")
print("per-country:", m.get("decision_tuning", {}).get("by_country"))

In [ ]:
# --- 14. stage 5: full test-set inference -> output/*.tsv -----------------------
# Kaggle gives ~30 GB RAM; the default query batch is safe. Lower --query-batch (e.g.
# 60000) if you hit a memory limit on the full run.
run(["scripts/06_infer.py", "--run-id", "kaggle", "--query-batch", "120000"],
    "full test inference")

In [ ]:
# --- 15. validate + preview the submittable TSV ---------------------------------
sub = subprocess.run(
    [sys.executable, os.path.join(CODE, "student_resource/utils/validate_submission.py"),
     "--matching", os.path.join(OUT, "matching_results.tsv"),
     "--candidate", os.path.join(OUT, "candidate_pairs.tsv"),
     "--test-dir", os.path.join(DATA_DIR, "test")],
    cwd=CODE, capture_output=True, text=True)
print(sub.stdout, sub.stderr)
print("=" * 60, "\nhead of matching_results.tsv:")
with open(os.path.join(OUT, "matching_results.tsv"), encoding="utf-8") as f:
    for i, line in zip(range(6), f):
        print(line.rstrip())

In [ ]:
# --- 16. surface the outputs in /kaggle/working for download --------------------
# Files under /kaggle/working attach to the notebook Output; download
# matching_results.tsv from the Output tab and upload it to the challenge portal.
for f in ("matching_results.tsv", "candidate_pairs.tsv"):
    src = os.path.join(OUT, f)
    if os.path.isfile(src):
        shutil.copy(src, os.path.join("/kaggle/working", f))
        print(f"{f}: {os.path.getsize(src)/1e6:.1f} MB -> /kaggle/working/{f}")
print("\nDONE — download matching_results.tsv from the Output tab.")